<a href="https://colab.research.google.com/github/TrSaleMane/municipality-ai-course/blob/main/docs/ipynb/day3_theme_customization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏛️ Day 3：テーマ別カスタマイズ演習
## 新人SE向け 自治体AIシステム開発 模擬体験講座

---

### このノートブックの目標

1. Day2で作ったRAGシステムをチームのテーマに合わせてカスタマイズする
2. オリジナル機能を1つ以上追加する
3. デモ発表できる状態に仕上げる

### チームテーマを選んでください

| テーマ | 担当業務 | ジャンプ先 |
|--------|---------|----------|
| 🏠 **A：住民窓口** | 転入届・各種申請の手続き案内 | Section 2-A |
| 🛣️ **B：土木・インフラ** | 道路損傷報告の受付・分類 | Section 2-B |
| 👨‍👩‍👧 **C：福祉・子育て** | 保育所申込・手当申請の案内 | Section 2-C |
| 🏫 **D：教育** | 就学案内・学校行事のQ&A | Section 2-D |

> 📌 Copyright © デジタル庁 / Licensed under MIT License

---
## Section 1：共通セットアップ（全チーム実行）

In [ ]:
!pip install -q google-generativeai chromadb pymupdf gradio

In [ ]:
import os
import chromadb
import gradio as gr
import google.generativeai as genai
from google.colab import userdata
from typing import List, Tuple
from datetime import datetime

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel('gemini-1.5-flash')

print('✅ セットアップ完了')

In [ ]:
# ======================================================
# Day2で作ったRAGの基底クラス（全テーマ共通）
# ======================================================

def split_into_chunks(text: str, chunk_size: int = 300, overlap: int = 50) -> List[str]:
    qa_pairs = [block.strip() for block in text.split('\n\n') if block.strip()]
    chunks = []
    for qa in qa_pairs:
        if len(qa) <= chunk_size:
            chunks.append(qa)
        else:
            for i in range(0, len(qa), chunk_size - overlap):
                chunk = qa[i:i + chunk_size]
                if chunk.strip():
                    chunks.append(chunk)
    return [c for c in chunks if len(c) > 20]


class BaseVectorStore:
    def __init__(self, collection_name: str):
        self.client = chromadb.Client()
        self.collection = self.client.get_or_create_collection(name=collection_name)

    def embed_text(self, text: str) -> List[float]:
        result = genai.embed_content(
            model='models/text-embedding-004',
            content=text,
            task_type='retrieval_document'
        )
        return result['embedding']

    def add_documents(self, chunks: List[str]):
        print(f'🔄 {len(chunks)}個のチャンクをベクトル化中...')
        embeddings = [self.embed_text(chunk) for chunk in chunks]
        self.collection.add(
            documents=chunks,
            embeddings=embeddings,
            ids=[f'chunk_{i}' for i in range(len(chunks))]
        )
        print(f'✅ {len(chunks)}個のチャンクをDBに登録完了')

    def search(self, query: str, n_results: int = 3) -> List[str]:
        query_embedding = genai.embed_content(
            model='models/text-embedding-004',
            content=query,
            task_type='retrieval_query'
        )['embedding']
        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=n_results
        )
        return results['documents'][0]

print('✅ 基底クラス読み込み完了')

---
## Section 2-A：🏠 住民窓口チーム
### テーマ：転入届・各種申請の手続き案内AI

**追加する機能（例）**
- 手続きの「必要書類チェックリスト」を自動生成する
- 窓口の混雑時間帯を考慮した来庁アドバイス

In [ ]:
# ======================================================
# 【テーマA】住民窓口 FAQデータ
# ここにチームのオリジナルFAQを追加してください
# ======================================================

THEME_A_FAQ = """
=== 転入・転出手続き ===

Q: 引越ししました。転入届はどうすればいいですか？
A: 引越し後14日以内に市役所1階 市民課で転入届を提出してください。
   必要書類：転出証明書（前住所の市区町村で発行）、本人確認書類（運転免許証・マイナンバーカード等）
   受付時間：平日8:30〜17:15、第2・第4土曜9:00〜12:00

Q: 転出証明書はどこで取得できますか？
A: 現在お住まいの市区町村の窓口で発行してもらえます。
   マイナンバーカードをお持ちの場合、転出届はマイナポータルからオンラインでも手続き可能です。
   その場合、転出証明書は不要です。

=== 印鑑登録 ===

Q: 印鑑登録をしたいのですが、どうすればいいですか？
A: 市役所1階 市民課の窓口で手続きできます。
   必要なもの：登録したい印鑑、本人確認書類
   手数料：300円
   即日発行可能です（本人申請の場合）。
   代理人申請の場合は、照会書を郵送するため数日かかります。

=== マイナンバーカード ===

Q: マイナンバーカードを申請したいです。
A: 以下の方法で申請できます。
   【オンライン】マイナポータル（https://myna.go.jp）から申請
   【郵便】通知カードの下部の申請書に写真を貼って郵送
   【窓口】市役所1階 市民課（要予約）
   申請から約1〜2ヶ月でカードが届きます。手数料は無料です。

=== 窓口混雑情報 ===

Q: 市役所の窓口が空いている時間帯はいつですか？
A: 比較的空いている時間帯は以下の通りです。
   【おすすめ】平日の10:00〜11:30、14:00〜16:00
   【混雑】月曜午前、月末・月初、引越しシーズン（3月〜4月）
   事前に電話予約できる手続きもあります。市民課（000-0000-0000）までお問い合わせください。

# ↓ チームオリジナルのQ&Aをここに追加してください
"""

print(f'✅ テーマAデータ準備完了（{len(THEME_A_FAQ)}文字）')

In [ ]:
# ======================================================
# 【テーマA】オリジナル機能：必要書類チェックリスト生成
# ======================================================

class WindowBotA(BaseVectorStore):
    """住民窓口AI（チェックリスト生成機能付き）"""

    SYSTEM_PROMPT = """
    あなたは市役所の住民窓口AIアシスタントです。
    転入届・印鑑登録・マイナンバーカードなどの手続きを専門に案内します。
    参考情報をもとに正確に回答し、必要書類は必ずリスト形式で案内してください。
    情報がない場合は「市民課（000-0000-0000）にお問い合わせください」と案内してください。
    """

    def __init__(self):
        super().__init__('theme_a_window')
        chunks = split_into_chunks(THEME_A_FAQ)
        self.add_documents(chunks)

    def answer(self, question: str) -> str:
        relevant = self.search(question, n_results=3)
        context = '\n\n'.join(relevant)
        prompt = f"{self.SYSTEM_PROMPT}\n\n=== 参考情報 ===\n{context}\n\n質問：{question}"
        return model.generate_content(prompt).text

    def generate_checklist(self, procedure: str) -> str:
        """手続き名から必要書類チェックリストを生成（オリジナル機能）"""
        relevant = self.search(procedure, n_results=3)
        context = '\n\n'.join(relevant)
        prompt = f"""
        以下の参考情報をもとに、「{procedure}」の必要書類チェックリストを作成してください。
        マークダウンのチェックボックス形式（- [ ] 書類名）で出力してください。

        参考情報：{context}
        """
        return model.generate_content(prompt).text


bot_a = WindowBotA()

# テスト
print('\n=== チェックリスト生成テスト ===')
print(bot_a.generate_checklist('転入届'))

---
## Section 2-B：🛣️ 土木・インフラチーム
### テーマ：道路損傷報告の受付・自動分類AI

**追加する機能（例）**
- 報告内容を「緊急度」で自動分類する（高/中/低）
- 報告票を自動フォーマットして出力する

In [ ]:
# ======================================================
# 【テーマB】土木・インフラ FAQデータ
# ======================================================

THEME_B_FAQ = """
=== 道路損傷・不具合報告 ===

Q: 道路に大きな穴（ポットホール）があります。どこに連絡しますか？
A: 市が管理する道路の場合は建設課道路維持係（000-0000-0001）までご連絡ください。
   緊急の場合（深さ10cm以上・幅30cm以上）は優先対応します。
   場所（住所・目標物）と損傷の大きさをお知らせください。
   県道の場合は県土木事務所、国道の場合は国道事務所が管轄です。

Q: 道路の白線（区画線）が消えかかっています。
A: 建設課道路維持係にご連絡ください。
   場所と消えている区間の長さをお知らせいただくとスムーズです。
   対応は通常2〜4週間以内を目安としています。

Q: ガードレールが壊れています。
A: 緊急度の高い案件として建設課道路維持係に連絡してください。
   二次事故防止のため、できれば近くに通行止めの表示などをお願いします。
   夜間・休日の緊急連絡先：市役所代表（000-0000-0000）

=== 道路工事申請 ===

Q: 道路を掘削する工事をしたいです。許可が必要ですか？
A: 市道を掘削する場合は「道路占用許可申請」が必要です。
   工事着手の2週間前までに建設課道路管理係に申請書を提出してください。
   必要書類：申請書、位置図、平面図、断面図、施工計画書

Q: 道路に仮設足場を設置したい場合はどうすればいいですか？
A: 道路占用許可申請が必要です。
   設置期間・面積によって占用料が発生します。
   詳しくは建設課道路管理係（000-0000-0002）にお問い合わせください。

=== 橋梁・インフラ管理 ===

Q: 橋に異常を見つけました。
A: 建設課維持管理係までご連絡ください。
   橋の名前（橋名板に記載）または場所をお知らせください。
   重大な損傷の場合は通行止め等の措置をとる場合があります。

# ↓ チームオリジナルのQ&Aをここに追加してください
"""

print(f'✅ テーマBデータ準備完了（{len(THEME_B_FAQ)}文字）')

In [ ]:
# ======================================================
# 【テーマB】オリジナル機能：緊急度自動分類 + 報告票生成
# ======================================================

class RoadDamageBot(BaseVectorStore):
    """道路損傷報告AI（緊急度分類・報告票生成機能付き）"""

    SYSTEM_PROMPT = """
    あなたは市役所建設課の道路損傷受付AIです。
    住民からの道路損傷・不具合の報告を受け付け、適切な担当部署を案内します。
    報告を受けたら必ず「場所」「損傷の種類」「大きさ」を確認してください。
    """

    URGENCY_LEVELS = {
        '高': '🔴 緊急対応（24時間以内）',
        '中': '🟡 優先対応（1週間以内）',
        '低': '🟢 通常対応（1ヶ月以内）'
    }

    def __init__(self):
        super().__init__('theme_b_road')
        chunks = split_into_chunks(THEME_B_FAQ)
        self.add_documents(chunks)

    def classify_urgency(self, report: str) -> dict:
        """報告内容から緊急度を自動判定（オリジナル機能）"""
        prompt = f"""
        以下の道路損傷報告の緊急度を判定してください。
        レスポンスはJSON形式で返してください。

        緊急度の基準：
        - 高：人身事故・通行不能のリスクあり（深い穴・ガードレール破損・陥没など）
        - 中：放置すると危険になる可能性あり（白線消え・舗装劣化など）
        - 低：軽微な損傷（小さなひび割れ・側溝の汚れなど）

        報告内容：{report}

        JSON形式で返答：
        {{"urgency": "高/中/低", "reason": "判定理由", "action": "推奨対応"}}
        """
        import json, re
        response = model.generate_content(prompt).text
        match = re.search(r'\{.*?\}', response, re.DOTALL)
        if match:
            return json.loads(match.group())
        return {'urgency': '中', 'reason': '判定不能', 'action': '窓口に確認してください'}

    def generate_report(self, location: str, damage_type: str, description: str) -> str:
        """道路損傷報告票を自動生成（オリジナル機能）"""
        urgency_result = self.classify_urgency(description)
        urgency_label = self.URGENCY_LEVELS.get(urgency_result['urgency'], '🟡 優先対応')
        now = datetime.now().strftime('%Y年%m月%d日 %H:%M')

        report = f"""
## 道路損傷報告票

| 項目 | 内容 |
|------|------|
| 受付日時 | {now} |
| 場所 | {location} |
| 損傷種別 | {damage_type} |
| 緊急度 | {urgency_label} |
| 判定理由 | {urgency_result['reason']} |
| 推奨対応 | {urgency_result['action']} |

### 詳細説明
{description}

### 担当部署
建設課道路維持係（000-0000-0001）
"""
        return report


bot_b = RoadDamageBot()

# テスト
print('\n=== 緊急度分類テスト ===')
result = bot_b.classify_urgency('国道沿いのガードレールが車の衝突でひしゃげています。今にも倒れそうです。')
print(result)

print('\n=== 報告票生成テスト ===')
report = bot_b.generate_report(
    location='○○市△△町1-2-3 ○○交差点付近',
    damage_type='ポットホール（路面の穴）',
    description='深さ約15cm、幅約40cmの穴があります。自転車が転倒しそうで危険です。'
)
print(report)

---
## Section 2-C：👨‍👩‍👧 福祉・子育てチーム
### テーマ：保育所申込・手当申請の案内AI

**追加する機能（例）**
- 子どもの年齢を入力すると受けられるサービスを一覧表示する
- 申請期限リマインダーメッセージを生成する

In [ ]:
# ======================================================
# 【テーマC】福祉・子育て FAQデータ
# ======================================================

THEME_C_FAQ = """
=== 保育所・幼稚園 ===

Q: 保育所に入所申請するにはどうすればいいですか？
A: 毎年10月〜11月に翌年4月入所の申し込みを受け付けます。
   窓口：こども課（市役所3階）または各保育所
   必要書類：入所申請書、就労証明書（保護者双方分）、マイナンバー書類
   選考結果は翌年2月頃に通知します。
   年度途中の入所は随時受け付けています（空き状況による）。

Q: 保育料はいくらですか？
A: 保育料は世帯収入によって異なります。
   3歳以上は幼児教育・保育の無償化により原則無料です。
   3歳未満は住民税額に応じて月額0円〜57,000円の範囲で設定されます。
   詳しくはこども課（000-0000-0003）にお問い合わせください。

=== 児童手当 ===

Q: 児童手当の支給額を教えてください。
A: 支給額（月額）は以下の通りです。
   0〜2歳：15,000円
   3歳〜小学校修了前：10,000円（第3子以降15,000円）
   中学生：10,000円
   ※所得制限あり（所得制限超過の場合は特例給付5,000円）
   支払いは2月・6月・10月の年3回です。

Q: 児童手当の申請はどうすればいいですか？
A: 出生・転入などの事由が発生した翌日から15日以内にこども課で申請してください。
   必要書類：認定請求書、請求者の健康保険証、通帳のコピー、マイナンバー書類
   申請が遅れると遡及支給を受けられない場合があります。お早めに手続きください。

=== 子育て支援サービス ===

Q: 一時保育は利用できますか？
A: 就労・通院・リフレッシュなどの理由で一時的に保育所を利用できます。
   対象：生後6ヶ月〜就学前のお子さん
   料金：1時間200円〜（施設によって異なる）
   事前予約が必要です。各保育所または子育て支援センターにお問い合わせください。

Q: 子育て支援センターではどんなことができますか？
A: 就学前のお子さんと保護者が自由に遊べるスペースを提供しています。
   育児相談・子育て講座・親子イベントなども開催しています。
   利用料は無料です。予約不要で平日9:00〜16:00に利用できます。

# ↓ チームオリジナルのQ&Aをここに追加してください
"""

print(f'✅ テーマCデータ準備完了（{len(THEME_C_FAQ)}文字）')

In [ ]:
# ======================================================
# 【テーマC】オリジナル機能：年齢別サービス一覧表示
# ======================================================

class ChildcareBot(BaseVectorStore):
    """子育て支援AI（年齢別サービス案内機能付き）"""

    SYSTEM_PROMPT = """
    あなたは市役所こども課の子育て支援AIアシスタントです。
    保育所・児童手当・子育て支援サービスについて丁寧に案内します。
    申請期限がある手続きは必ず期限を強調してください。
    """

    def __init__(self):
        super().__init__('theme_c_childcare')
        chunks = split_into_chunks(THEME_C_FAQ)
        self.add_documents(chunks)

    def answer(self, question: str) -> str:
        relevant = self.search(question, n_results=3)
        context = '\n\n'.join(relevant)
        prompt = f"{self.SYSTEM_PROMPT}\n\n=== 参考情報 ===\n{context}\n\n質問：{question}"
        return model.generate_content(prompt).text

    def get_services_by_age(self, age_months: int) -> str:
        """子どもの月齢に合わせた利用可能サービスを案内（オリジナル機能）"""
        age_str = f'{age_months // 12}歳{age_months % 12}ヶ月' if age_months >= 12 else f'{age_months}ヶ月'
        prompt = f"""
        以下の参考情報をもとに、{age_str}のお子さんが利用できる市のサービスを
        わかりやすく一覧にしてください。
        マークダウンの表形式で「サービス名」「内容」「申請先」を示してください。

        参考情報：{THEME_C_FAQ}
        """
        return model.generate_content(prompt).text


bot_c = ChildcareBot()

# テスト
print('=== 年齢別サービス案内テスト ===')
print(bot_c.get_services_by_age(8))  # 8ヶ月

---
## Section 2-D：🏫 教育チーム
### テーマ：就学案内・学校行事のQ&A AI

**追加する機能（例）**
- 就学に必要な手続きをステップ形式で案内する
- 転校手続きのフローを自動生成する

In [ ]:
# ======================================================
# 【テーマD】教育 FAQデータ
# ======================================================

THEME_D_FAQ = """
=== 就学手続き ===

Q: 子どもが小学校に入学するにはどうすればいいですか？
A: 毎年10〜11月頃に就学通知書を郵送します。指定された学校に通知書を持参してください。
   就学時健康診断（11月頃）への参加が必要です。
   入学式は翌年4月上旬です。学用品等の準備物は入学説明会（2〜3月）でお知らせします。
   手続き窓口：学校教育課（市役所4階）

Q: 指定校以外の学校に通わせたい場合はどうすればいいですか？
A: 区域外就学の申請が必要です。
   理由（通学路の安全・兄弟の通学校への統一など）を記載した申請書を学校教育課に提出してください。
   認められるかどうかは理由と受け入れ校の状況によります。

=== 転校手続き ===

Q: 市内で引越しする場合、転校の手続きはどうすればいいですか？
A: ①転入届を市民課で手続き
   ②現在の学校から「在学証明書」と「教科書給与証明書」を受け取る
   ③学校教育課で転校の手続き（就学校指定変更申請）
   ④新しい学校へ書類を持参して入学手続き
   学期途中の転校も可能です。できるだけ早めにご相談ください。

Q: 他市から転校してきます。何が必要ですか？
A: 前の市区町村の学校から以下の書類を受け取ってください。
   ・在学証明書
   ・教科書給与証明書
   転入届後に学校教育課で就学校の指定を受け、指定校に書類を持参してください。

=== 就学援助 ===

Q: 就学援助制度とはどのような制度ですか？
A: 経済的な理由で学用品・給食費・修学旅行費などの負担が難しいご家庭を支援する制度です。
   対象：生活保護を受けているご家庭、または収入が一定基準以下のご家庭
   申請窓口：学校教育課または各学校
   年度ごとに申請が必要です。詳しくはお問い合わせください。

# ↓ チームオリジナルのQ&Aをここに追加してください
"""

print(f'✅ テーマDデータ準備完了（{len(THEME_D_FAQ)}文字）')

In [ ]:
# ======================================================
# 【テーマD】オリジナル機能：転校フロー自動生成
# ======================================================

class EducationBot(BaseVectorStore):
    """学校教育AI（転校フロー生成機能付き）"""

    SYSTEM_PROMPT = """
    あなたは市役所学校教育課のAIアシスタントです。
    就学・転校・就学援助などの手続きを丁寧に案内します。
    手続きが複数のステップにわたる場合は、必ずステップ形式で説明してください。
    """

    def __init__(self):
        super().__init__('theme_d_education')
        chunks = split_into_chunks(THEME_D_FAQ)
        self.add_documents(chunks)

    def answer(self, question: str) -> str:
        relevant = self.search(question, n_results=3)
        context = '\n\n'.join(relevant)
        prompt = f"{self.SYSTEM_PROMPT}\n\n=== 参考情報 ===\n{context}\n\n質問：{question}"
        return model.generate_content(prompt).text

    def generate_transfer_flow(self, situation: str) -> str:
        """転校フローチャートをテキストで生成（オリジナル機能）"""
        prompt = f"""
        以下の参考情報をもとに、「{situation}」の場合の転校手続きフローを
        番号付きステップ形式でわかりやすく説明してください。
        各ステップに「担当窓口」と「必要書類」も含めてください。

        参考情報：{THEME_D_FAQ}
        """
        return model.generate_content(prompt).text


bot_d = EducationBot()

# テスト
print('=== 転校フロー生成テスト ===')
print(bot_d.generate_transfer_flow('市内で引越しして小学校を転校する'))

---
## Section 3：チームのGradio UIを完成させる

選んだテーマのボットを使ってGradio UIを構築してください。

In [ ]:
# ======================================================
# 【全チーム共通】Gradio UI テンプレート
# 使うボットを切り替えてください
# ======================================================

# 使うボットをここで選択
# active_bot = bot_a  # A：住民窓口チーム
# active_bot = bot_b  # B：土木・インフラチーム
# active_bot = bot_c  # C：福祉・子育てチーム
# active_bot = bot_d  # D：教育チーム
active_bot = bot_a   # ← ここを変更してください

TEAM_CONFIG = {
    'title': '🏛️ 市役所AIアシスタント（Day3 カスタム版）',
    'description': 'チームのオリジナルAIシステムです。',
    'examples': [
        '手続きについて教えてください',  # ← チームに合わせて変更
        '必要な書類は何ですか？',
        '窓口はどこですか？'
    ]
}


def team_chat(message: str, history: list) -> str:
    if not message.strip():
        return '質問を入力してください。'
    return active_bot.answer(message)


with gr.Blocks(title=TEAM_CONFIG['title'], theme=gr.themes.Soft()) as team_demo:
    gr.Markdown(f"# {TEAM_CONFIG['title']}\n{TEAM_CONFIG['description']}")
    gr.Markdown('> ⚠️ これは講座演習用のデモです。実際の手続きは市役所にご確認ください。')

    chatbot = gr.ChatInterface(
        fn=team_chat,
        examples=TEAM_CONFIG['examples'],
        title=''
    )

team_demo.launch(share=True)

---
## 🎯 発表前チェックリスト

発表前に以下を確認してください。

- [ ] チームのテーマに合ったFAQデータをオリジナルで追加した
- [ ] オリジナル機能が1つ以上動いている
- [ ] GradioのUIが起動してチャットできる
- [ ] 5つ以上の質問で正しく回答できることを確認した
- [ ] 回答できない質問（範囲外）を適切に断れる
- [ ] 発表資料（presentation_template.md）が完成している

---

## Day 3 まとめ

| 学んだこと | ポイント |
|-----------|--------|
| テーマ特化RAG | 専門分野のFAQを充実させるほど精度が上がる |
| オリジナル機能 | LLMの出力をJSON化→構造化データとして活用できる |
| システム設計 | 「なぜその設計にしたか」を言語化することが重要 |
| 自治体AI特有の考慮点 | 個人情報・セキュリティ・正確性の担保が必須 |